# NVDA Regimes, Signals & Volatility Plan

**Original Question:** How do NVDA's market regimes differ, which technical indicators are most profitable in each regime, and what is the forecast for the next 30 days of volatility?

_Exported from PlotStudio_

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


# PlotStudio stub — no-op outside the app
def add_to_workspace(df):
    pass


# Load source data
df_nvda = pd.read_csv("data/df_nvda.csv")
df_nvda_base = pd.read_csv("data/df_nvda_base.csv")
df_nvda_feat = pd.read_csv("data/df_nvda_feat.csv")

## Task 1: Create a clean, return-based and indicator-enriched NVDA time series suitable for regime detection, strategy backtesting, and volatility modeling.

**Acceptance Criteria:** A single primary analysis DataFrame exists with datetime index, log returns, rolling volatility, and all core technical indicators computed, along with at least one candlestick-style visualization of the price history with trend context.

### 1.1: Ensure df_nvda is properly typed and ordered for time series analysis and derive core log-return series.
_Output: print_

In [ ]:
import numpy as np

# Use existing df_nvda, do not recreate or reload
# Create df_nvda_base with required transformations

df_nvda_base = df_nvda.copy()
df_nvda_base["date"] = pd.to_datetime(df_nvda_base["date"], errors="coerce")
df_nvda_base = df_nvda_base.sort_values("date").reset_index(drop=True)

# Compute close-to-close log return
# log_return = log(close_t / close_{t-1})
df_nvda_base["log_return_close"] = np.log(
    df_nvda_base["close"] / df_nvda_base["close"].shift(1)
)

# Compute intraday log return: log(close / open)
df_nvda_base["log_return_intraday"] = np.log(
    df_nvda_base["close"] / df_nvda_base["open"]
)

# Check for missing values in new columns
missing_close = df_nvda_base["log_return_close"].isna().sum()
missing_intraday = df_nvda_base["log_return_intraday"].isna().sum()

# Check date range and shape
start_date = df_nvda_base["date"].min()
end_date = df_nvda_base["date"].max()
shape = df_nvda_base.shape

# Check for unexpected calendar gaps (more than 3 days between consecutive dates)
date_diffs = df_nvda_base["date"].diff().dt.days.dropna()
gap_threshold = 3
# Align index for boolean indexing
mask_gaps = date_diffs > gap_threshold
gaps = df_nvda_base.loc[mask_gaps.index[mask_gaps], ["date"]]

print(f"df_nvda_base shape: {shape}")
print(f"Date range: {start_date.date()} to {end_date.date()}")
print(f"Missing values in log_return_close: {missing_close}")
print(f"Missing values in log_return_intraday: {missing_intraday}")
if not gaps.empty:
    print("Unexpected calendar gaps (more than 3 days between trading days):")
    print(gaps)
else:
    print("No unexpected calendar gaps detected (beyond normal trading breaks).")

add_to_workspace(df_nvda_base)


### 1.2: Quantify realized volatility over multiple horizons to support regime detection and later benchmarking of forecasted volatility.
_Output: print_

In [ ]:
# Use existing df_nvda_feat, which already contains the required rolling volatility columns
# Print compact descriptive statistics for these volatility columns
import numpy as np

windows = [10, 20, 60]
vol_cols = [f"realized_vol_{w}d" for w in windows] + [
    f"realized_vol_{w}d_ann" for w in windows
]

# Compute descriptive statistics for each volatility column
stats = []
for col in vol_cols:
    series = df_nvda_feat[col]
    stats.append(
        {
            "column": col,
            "count": series.count(),
            "mean": series.mean(),
            "std": series.std(),
            "min": series.min(),
            "max": series.max(),
            "missing": series.isna().sum(),
        }
    )

import pandas as pd

df_vol_stats = pd.DataFrame(stats)

print("Descriptive statistics for realized volatility columns:")
print(df_vol_stats[["column", "count", "mean", "std", "min", "max", "missing"]])

add_to_workspace(df_nvda_feat)


### 1.3: Augment the series with a comprehensive set of standard technical indicators across trend, momentum, volatility, trend strength, and volume confirmation families.
_Output: print_

In [ ]:
import pandas_ta_classic as ta
import numpy as np

# Work on a copy to avoid modifying df_nvda_feat
# All indicator columns will be added to this DataFrame
# Columns to add: SMA_10, SMA_20, SMA_50, SMA_200 (on close), RSI_14, MACD (line/signal/hist), Bollinger Bands (20d), ADX_14 (+DMP, -DMN), OBV

df_nvda_feat_indicators = df_nvda_feat.copy()

# Simple Moving Averages on close
for length in [10, 20, 50, 200]:
    df_nvda_feat_indicators[f"SMA_{length}"] = df_nvda_feat_indicators.ta.sma(
        length=length, close="close"
    )

# RSI(14)
df_nvda_feat_indicators["RSI_14"] = df_nvda_feat_indicators.ta.rsi(
    length=14, close="close"
)

# MACD (default fast=12, slow=26, signal=9)
df_macd = df_nvda_feat_indicators.ta.macd(fast=12, slow=26, signal=9, close="close")
# MACD returns columns: MACD_12_26_9, MACDh_12_26_9, MACDs_12_26_9
for col in df_macd.columns:
    df_nvda_feat_indicators[col] = df_macd[col]

# Bollinger Bands (20d, 2 std)
df_bbands = df_nvda_feat_indicators.ta.bbands(length=20, std=2, close="close")
# BBL_20_2.0, BBM_20_2.0, BBU_20_2.0
for col in df_bbands.columns:
    df_nvda_feat_indicators[col] = df_bbands[col]

# ADX(14) with DMP/DMN
df_adx = df_nvda_feat_indicators.ta.adx(
    length=14, high="high", low="low", close="close"
)
# ADX_14, DMP_14, DMN_14
for col in df_adx.columns:
    df_nvda_feat_indicators[col] = df_adx[col]

# OBV (On-Balance Volume)
df_nvda_feat_indicators["OBV"] = df_nvda_feat_indicators.ta.obv(
    close="close", volume="volume"
)

# Print the latest 5 rows for close and all indicator columns to verify
indicator_cols = [
    "close",
    "SMA_10",
    "SMA_20",
    "SMA_50",
    "SMA_200",
    "RSI_14",
    "MACD_12_26_9",
    "MACDh_12_26_9",
    "MACDs_12_26_9",
    "BBL_20_2.0",
    "BBM_20_2.0",
    "BBU_20_2.0",
    "ADX_14",
    "DMP_14",
    "DMN_14",
    "OBV",
]
print("Latest 5 rows of close and technical indicators:")
print(df_nvda_feat_indicators[indicator_cols].tail())

add_to_workspace(df_nvda_feat_indicators)


### 1.4: Provide visual context of NVDA’s price evolution and broad trend regimes using a candlestick chart with key moving averages.
_Output: figure_

In [ ]:
import plotly.graph_objects as go

# Prepare traces for candlestick and moving averages
fig = go.Figure()

# Candlestick trace
fig.add_trace(
    go.Candlestick(
        x=df_nvda_feat_indicators["date"],
        open=df_nvda_feat_indicators["open"],
        high=df_nvda_feat_indicators["high"],
        low=df_nvda_feat_indicators["low"],
        close=df_nvda_feat_indicators["close"],
        name="NVDA OHLC",
        increasing_line_color="green",
        decreasing_line_color="red",
        showlegend=True,
    )
)

# 50-day SMA
fig.add_trace(
    go.Scatter(
        x=df_nvda_feat_indicators["date"],
        y=df_nvda_feat_indicators["SMA_50"],
        mode="lines",
        line=dict(color="blue", width=2),
        name="SMA 50d",
    )
)

# 200-day SMA
fig.add_trace(
    go.Scatter(
        x=df_nvda_feat_indicators["date"],
        y=df_nvda_feat_indicators["SMA_200"],
        mode="lines",
        line=dict(color="orange", width=2),
        name="SMA 200d",
    )
)

fig.update_layout(
    title="NVDA Price History: Bull Runs, Pullbacks, and Consolidation (Candlestick with 50d/200d SMA)",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    xaxis_rangeslider_visible=False,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.show()


## Task 2: Segment NVDA’s history into a small number of interpretable market regimes based on realized volatility and trend characteristics, and describe each regime’s behavior over time.

**Acceptance Criteria:** Each trading day is assigned to a regime label; regimes are summarized by typical volatility, trend strength, and duration; and at least one visualization shows how regimes unfold over time.

_Mode: exploratory_

### 2.1: Create a compact feature set per day capturing volatility, trend, and trend strength for clustering into regimes.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Start from the indicator-enriched DataFrame
# Required features:
# - 20-day rolling mean of log_return_close (local trend)
# - realized_vol_20d, realized_vol_60d (already present)
# - ADX_14, DMP_14, DMN_14 (already present)
# - normalized distance above 200d SMA: (close - SMA_200) / SMA_200

# Work on a copy to avoid modifying the original
df_nvda_regime_features = df_nvda_feat_indicators.copy()

# 20-day rolling mean of log_return_close
# (local trend, centered on most recent 20 days)
df_nvda_regime_features["trend_20d_mean"] = (
    df_nvda_regime_features["log_return_close"]
    .rolling(window=20, min_periods=20)
    .mean()
)

# Normalized distance above 200d SMA
# (close - SMA_200) / SMA_200
df_nvda_regime_features["dist_above_SMA200"] = (
    df_nvda_regime_features["close"] - df_nvda_regime_features["SMA_200"]
) / df_nvda_regime_features["SMA_200"]

# Select only rows where all regime features are present (no missing in any feature window)
feature_cols = [
    "date",
    "trend_20d_mean",
    "realized_vol_20d",
    "realized_vol_60d",
    "ADX_14",
    "DMP_14",
    "DMN_14",
    "dist_above_SMA200",
]

# Filter to rows with complete feature windows (no missing in any of the above)
df_nvda_regime_features = (
    df_nvda_regime_features[feature_cols].dropna().reset_index(drop=True)
)

# Print compact summary
row_count = len(df_nvda_regime_features)
date_min = df_nvda_regime_features["date"].min()
date_max = df_nvda_regime_features["date"].max()
missing_by_col = df_nvda_regime_features.isna().sum()
means = df_nvda_regime_features.drop(columns=["date"]).mean()
stds = df_nvda_regime_features.drop(columns=["date"]).std()

print(f"df_nvda_regime_features row count: {row_count}")
print(f"Date span: {date_min.date()} to {date_max.date()}")
print("Missing values after filtering (should all be 0):")
print(missing_by_col)
print("Feature means:")
print(means)
print("Feature stds:")
print(stds)

add_to_workspace(df_nvda_regime_features)


### 2.2: Assign each day to one of a small number of regimes defined by volatility and trend features.
_Output: exploration_

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Use the regime feature matrix
feature_cols = [
    "trend_20d_mean",
    "realized_vol_20d",
    "realized_vol_60d",
    "ADX_14",
    "DMP_14",
    "DMN_14",
    "dist_above_SMA200",
]

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(df_nvda_regime_features[feature_cols])

# Evaluate k-means for k=3,4,5
results = []
for k in [3, 4, 5]:
    kmeans = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = kmeans.fit_predict(X)
    inertia = kmeans.inertia_
    sil = silhouette_score(X, labels)
    results.append({"k": k, "inertia": inertia, "silhouette": sil})

df_kmeans_eval = pd.DataFrame(results)
print("K-means regime clustering evaluation:")
print(df_kmeans_eval)

# Choose k=3 for interpretability (smallest k with reasonable separation)
k_best = 3
kmeans_final = KMeans(n_clusters=k_best, n_init=20, random_state=42)
labels_final = kmeans_final.fit_predict(X)

# Centroids in standardized space, convert back to original units
centroids_std = kmeans_final.cluster_centers_
centroids_orig = scaler.inverse_transform(centroids_std)
df_centroids = pd.DataFrame(centroids_orig, columns=feature_cols)
df_centroids["regime"] = [f"Regime {i}" for i in range(k_best)]
df_centroids = df_centroids.set_index("regime")

print("\nChosen regime centroids (original feature units):")
print(df_centroids)

# Short interpretation for each regime
for i, row in df_centroids.iterrows():
    trend = row["trend_20d_mean"]
    vol = row["realized_vol_20d"]
    adx = row["ADX_14"]
    dist = row["dist_above_SMA200"]
    desc = []
    if trend > 0.002:
        desc.append("Uptrend")
    elif trend < -0.002:
        desc.append("Downtrend")
    else:
        desc.append("Sideways/Flat")
    if vol > 0.035:
        desc.append("High Volatility")
    elif vol < 0.025:
        desc.append("Low Volatility")
    else:
        desc.append("Moderate Volatility")
    if adx > 30:
        desc.append("Strong Trend")
    elif adx < 20:
        desc.append("Weak Trend")
    else:
        desc.append("Moderate Trend")
    if dist > 0.2:
        desc.append("Well Above SMA200")
    elif dist < -0.05:
        desc.append("Below SMA200")
    else:
        desc.append("Near SMA200")
    print(f"{i}: {', '.join(desc)}")

# Merge regime labels back onto indicator-enriched NVDA history by date
# Create df_nvda_with_regime
# First, create a DataFrame with date and regime label
regime_labels = pd.DataFrame(
    {"date": df_nvda_regime_features["date"], "regime_label": labels_final}
)

# Merge onto df_nvda_feat_indicators
# Use left join to preserve all rows in df_nvda_feat_indicators
# (regime_label will be NaN for dates outside regime feature window)
df_nvda_with_regime = pd.merge(
    df_nvda_feat_indicators, regime_labels, on="date", how="left"
).reset_index(drop=True)

add_to_workspace(df_nvda_with_regime)


### 2.3: Describe how regimes differ in statistics and how they evolve over the historical timeline.
_Output: figure_

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Use df_regime, which contains all indicator columns and regime_label
# Only analyze rows with a valid regime_label (not NaN)
# Compute normalized distance above SMA_200 within this step

df_regime = df_nvda_with_regime.dropna(subset=["regime_label"]).copy()
df_regime["regime_label"] = df_regime["regime_label"].astype(int)

# Compute normalized distance above SMA_200: (close - SMA_200) / SMA_200
# SMA_200 may have NaNs at the start, but regime windows already filtered for valid features
# For summary, compute mean over available values

df_regime["dist_above_SMA200"] = (
    df_regime["close"] - df_regime["SMA_200"]
) / df_regime["SMA_200"]

# Regime summary: counts, share, mean/std log_return_close, mean realized_vol_20d/60d, mean ADX_14, mean dist_above_SMA200
grouped = df_regime.groupby("regime_label")
total = len(df_regime)
df_regime_summary = grouped.agg(
    days=("date", "count"),
    pct_of_days=("date", lambda x: len(x) / total),
    mean_log_return_close=("log_return_close", "mean"),
    std_log_return_close=("log_return_close", "std"),
    mean_realized_vol_20d=("realized_vol_20d", "mean"),
    mean_realized_vol_60d=("realized_vol_60d", "mean"),
    mean_ADX_14=("ADX_14", "mean"),
    mean_dist_above_SMA200=("dist_above_SMA200", "mean"),
).reset_index()

print(
    "NVDA Regime Summary (counts, share, returns, volatility, trend, distance above SMA200):"
)
print(df_regime_summary)


# Run-length statistics: for each regime, compute mean, max, and median consecutive run length
def compute_run_lengths(labels):
    run_lengths = {regime: [] for regime in np.unique(labels)}
    prev = None
    count = 0
    for label in labels:
        if label == prev:
            count += 1
        else:
            if prev is not None:
                run_lengths[prev].append(count)
            count = 1
            prev = label
    if prev is not None:
        run_lengths[prev].append(count)
    return run_lengths


run_lengths = compute_run_lengths(df_regime["regime_label"].values)
run_stats = []
for regime, runs in run_lengths.items():
    run_stats.append(
        {
            "regime_label": regime,
            "mean_run_length": np.mean(runs),
            "median_run_length": np.median(runs),
            "max_run_length": np.max(runs),
            "num_runs": len(runs),
        }
    )
df_run_stats = pd.DataFrame(run_stats)

print("\nConsecutive Run-Length Statistics by Regime:")
print(df_run_stats)

# Plot: close price over time, color by regime_label
regime_color_map = {0: "red", 1: "blue", 2: "green"}
fig = px.scatter(
    df_regime,
    x="date",
    y="close",
    color="regime_label",
    color_discrete_map=regime_color_map,
    title="NVDA Close Price by Market Regime: Regime Shifts Highlighted",
    labels={"close": "Close Price (USD)", "date": "Date", "regime_label": "Regime"},
)
fig.update_traces(marker=dict(size=6, line=dict(width=0)))
fig.update_traces(mode="markers")
fig.update_layout(legend_title_text="Regime")
fig.show()


## Task 3: Evaluate which technical indicator-based strategies deliver the best risk-adjusted performance within each identified regime and how their performance changes around regime transitions.

**Acceptance Criteria:** Daily strategy return series are computed for each indicator family, mapped to regimes, and summarized with Sharpe, Sortino, max drawdown, and VaR by regime so that top-performing indicators per regime can be clearly identified.

### 3.1: Translate key indicators into clear, rule-based long/flat (or long/short) position signals for backtesting.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Work on a copy to avoid modifying df_nvda_with_regime
# All signals must be pandas Series before shifting

df_nvda_signals_v2 = df_nvda_with_regime.copy()


# Helper: shift all signals by 1 to ensure only prior-close info is used
def shift_signal(series):
    return series.shift(1)


# SMA 50/200 signals
# Long-flat: 1 if SMA_50 > SMA_200, else 0
sma_long_flat = (df_nvda_signals_v2["SMA_50"] > df_nvda_signals_v2["SMA_200"]).astype(
    int
)
df_nvda_signals_v2["sig_SMA50_200_long_flat"] = shift_signal(sma_long_flat)
# Long-short: 1 if SMA_50 > SMA_200, -1 if SMA_50 < SMA_200, 0 otherwise
sma_long_short = pd.Series(
    np.where(
        df_nvda_signals_v2["SMA_50"] > df_nvda_signals_v2["SMA_200"],
        1,
        np.where(df_nvda_signals_v2["SMA_50"] < df_nvda_signals_v2["SMA_200"], -1, 0),
    ),
    index=df_nvda_signals_v2.index,
)
df_nvda_signals_v2["sig_SMA50_200_long_short"] = shift_signal(sma_long_short)

# RSI(14) signals
# Long-flat: 1 if RSI < 30 (oversold), 0 otherwise
rsi_long_flat = (df_nvda_signals_v2["RSI_14"] < 30).astype(int)
df_nvda_signals_v2["sig_RSI14_oversold_long_flat"] = shift_signal(rsi_long_flat)
# Long-short: 1 if RSI < 30, -1 if RSI > 70 (overbought), 0 otherwise
rsi_long_short = pd.Series(
    np.where(
        df_nvda_signals_v2["RSI_14"] < 30,
        1,
        np.where(df_nvda_signals_v2["RSI_14"] > 70, -1, 0),
    ),
    index=df_nvda_signals_v2.index,
)
df_nvda_signals_v2["sig_RSI14_oversold_overbought_long_short"] = shift_signal(
    rsi_long_short
)

# MACD signals
# Long-flat: 1 if MACD line > MACD signal, 0 otherwise
macd_long_flat = (
    df_nvda_signals_v2["MACD_12_26_9"] > df_nvda_signals_v2["MACDs_12_26_9"]
).astype(int)
df_nvda_signals_v2["sig_MACD_line_above_signal_long_flat"] = shift_signal(
    macd_long_flat
)
# Long-short: 1 if MACD line > MACD signal, -1 if <, 0 otherwise
macd_long_short = pd.Series(
    np.where(
        df_nvda_signals_v2["MACD_12_26_9"] > df_nvda_signals_v2["MACDs_12_26_9"],
        1,
        np.where(
            df_nvda_signals_v2["MACD_12_26_9"] < df_nvda_signals_v2["MACDs_12_26_9"],
            -1,
            0,
        ),
    ),
    index=df_nvda_signals_v2.index,
)
df_nvda_signals_v2["sig_MACD_line_vs_signal_long_short"] = shift_signal(macd_long_short)

# Bollinger Bands signals
# Long-flat: 1 if close < lower band (BBL_20_2.0), 0 otherwise
bb_long_flat = (df_nvda_signals_v2["close"] < df_nvda_signals_v2["BBL_20_2.0"]).astype(
    int
)
df_nvda_signals_v2["sig_BB_lower_long_flat"] = shift_signal(bb_long_flat)
# Long-short: 1 if close < lower band, -1 if close > upper band, 0 otherwise
bb_long_short = pd.Series(
    np.where(
        df_nvda_signals_v2["close"] < df_nvda_signals_v2["BBL_20_2.0"],
        1,
        np.where(df_nvda_signals_v2["close"] > df_nvda_signals_v2["BBU_20_2.0"], -1, 0),
    ),
    index=df_nvda_signals_v2.index,
)
df_nvda_signals_v2["sig_BB_meanrev_long_short"] = shift_signal(bb_long_short)

# ADX-filtered SMA trend signal
# Only take SMA50>200 long if ADX_14 > 25 (strong trend)
adx_filter = df_nvda_signals_v2["ADX_14"] > 25
adx_sma_long = (
    (df_nvda_signals_v2["SMA_50"] > df_nvda_signals_v2["SMA_200"]) & adx_filter
).astype(int)
df_nvda_signals_v2["sig_SMA50_200_ADXtrend_long_flat"] = shift_signal(adx_sma_long)

# OBV-based volume confirmation signal
# 5-day rolling OBV change: 1 if OBV change > 0, -1 if < 0, 0 otherwise
obv_5d_chg = df_nvda_signals_v2["OBV"].diff(5)
obv_signal = pd.Series(
    np.where(obv_5d_chg > 0, 1, np.where(obv_5d_chg < 0, -1, 0)),
    index=df_nvda_signals_v2.index,
)
df_nvda_signals_v2["sig_OBV_5d_change_long_short"] = shift_signal(obv_signal)

# List of all signal columns
signal_cols = [
    "sig_SMA50_200_long_flat",
    "sig_SMA50_200_long_short",
    "sig_RSI14_oversold_long_flat",
    "sig_RSI14_oversold_overbought_long_short",
    "sig_MACD_line_above_signal_long_flat",
    "sig_MACD_line_vs_signal_long_short",
    "sig_BB_lower_long_flat",
    "sig_BB_meanrev_long_short",
    "sig_SMA50_200_ADXtrend_long_flat",
    "sig_OBV_5d_change_long_short",
]

# Activity summary: for each signal, non-null count, active-day share (|position|>0), average position
summary = []
for col in signal_cols:
    series = df_nvda_signals_v2[col]
    non_null = series.notna().sum()
    active_share = (series.abs() > 0).mean()
    avg_pos = series.mean()
    summary.append(
        {
            "signal": col,
            "non_null_count": non_null,
            "active_day_share": active_share,
            "avg_position": avg_pos,
        }
    )
df_signal_summary_v2 = pd.DataFrame(summary)

print("Signal activity summary (non-null count, active-day share, avg position):")
print(df_signal_summary_v2)

add_to_workspace(df_nvda_signals_v2)


### 3.2: Translate signals into daily strategy returns and compute core risk and performance metrics overall and by regime.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use existing df_nvda_signals_v2, which contains all signals, log_return_close, and regime_label
# List of signal columns
signal_cols = [
    "sig_SMA50_200_long_flat",
    "sig_SMA50_200_long_short",
    "sig_RSI14_oversold_long_flat",
    "sig_RSI14_oversold_overbought_long_short",
    "sig_MACD_line_above_signal_long_flat",
    "sig_MACD_line_vs_signal_long_short",
    "sig_BB_lower_long_flat",
    "sig_BB_meanrev_long_short",
    "sig_SMA50_200_ADXtrend_long_flat",
    "sig_OBV_5d_change_long_short",
]

# Prepare output list
metrics = []

# Helper functions for metrics


def geometric_cum_return(log_returns):
    # log_returns: array-like
    return np.exp(np.nansum(log_returns)) - 1


def annualized_return(log_returns, n_days):
    # n_days: number of non-null days
    if n_days == 0:
        return np.nan
    mean_daily = np.nanmean(log_returns)
    return np.exp(mean_daily * 252) - 1


def annualized_vol(log_returns):
    return np.nanstd(log_returns, ddof=1) * np.sqrt(252)


def sharpe_ratio(log_returns):
    vol = np.nanstd(log_returns, ddof=1)
    mean = np.nanmean(log_returns)
    if vol == 0 or np.isnan(vol):
        return np.nan
    return mean / vol * np.sqrt(252)


def sortino_ratio(log_returns):
    # Downside deviation: std of negative returns only
    downside = log_returns[log_returns < 0]
    if len(downside) == 0:
        return np.nan
    downside_std = np.std(downside, ddof=1)
    mean = np.nanmean(log_returns)
    if downside_std == 0 or np.isnan(downside_std):
        return np.nan
    return mean / downside_std * np.sqrt(252)


def max_drawdown(log_returns):
    # Compute cumulative log return, then drawdown
    cum = np.nancumsum(log_returns)
    cum_exp = np.exp(cum)
    running_max = np.maximum.accumulate(cum_exp)
    drawdown = cum_exp / running_max - 1
    return np.min(drawdown)


def daily_var(log_returns, alpha=0.05):
    # 95% VaR (1-day): negative quantile
    if len(log_returns) == 0:
        return np.nan
    return -np.nanquantile(log_returns, alpha)


# For regime-level slicing, use regime_label (float with some NaN)
# Only consider rows where regime_label is not null for regime-level stats

df = df_nvda_signals_v2.copy()

# For each signal, compute metrics overall and by regime
for sig in signal_cols:
    # Overall metrics
    pos = df[sig]
    ret = df["log_return_close"]
    strat_ret = pos * ret
    valid = pos.notna() & ret.notna()
    strat_ret_valid = strat_ret[valid]
    pos_valid = pos[valid]
    n_days = valid.sum()
    active_share = (pos_valid.abs() > 0).mean() if n_days > 0 else np.nan
    metrics.append(
        {
            "strategy": sig,
            "regime_label": "All",
            "n_days": n_days,
            "cum_return": geometric_cum_return(strat_ret_valid),
            "ann_return": annualized_return(strat_ret_valid, n_days),
            "ann_vol": annualized_vol(strat_ret_valid),
            "sharpe": sharpe_ratio(strat_ret_valid),
            "sortino": sortino_ratio(strat_ret_valid),
            "max_drawdown": max_drawdown(strat_ret_valid),
            "daily_95pct_VaR": daily_var(strat_ret_valid, 0.05),
            "active_day_share": active_share,
        }
    )
    # Regime-level metrics
    for regime in sorted(df["regime_label"].dropna().unique()):
        regime_mask = df["regime_label"] == regime
        valid_regime = valid & regime_mask
        strat_ret_regime = strat_ret[valid_regime]
        pos_regime = pos[valid_regime]
        n_days_regime = valid_regime.sum()
        active_share_regime = (
            (pos_regime.abs() > 0).mean() if n_days_regime > 0 else np.nan
        )
        metrics.append(
            {
                "strategy": sig,
                "regime_label": int(regime),
                "n_days": n_days_regime,
                "cum_return": geometric_cum_return(strat_ret_regime),
                "ann_return": annualized_return(strat_ret_regime, n_days_regime),
                "ann_vol": annualized_vol(strat_ret_regime),
                "sharpe": sharpe_ratio(strat_ret_regime),
                "sortino": sortino_ratio(strat_ret_regime),
                "max_drawdown": max_drawdown(strat_ret_regime),
                "daily_95pct_VaR": daily_var(strat_ret_regime, 0.05),
                "active_day_share": active_share_regime,
            }
        )

# Create tidy DataFrame

df_strategy_metrics_v2 = pd.DataFrame(metrics)

# Sort by regime_label (All first, then 0,1,2), then by Sharpe descending within regime
regime_order = ["All", 0, 1, 2]
df_strategy_metrics_v2["regime_label"] = pd.Categorical(
    df_strategy_metrics_v2["regime_label"], categories=regime_order, ordered=True
)
df_strategy_metrics_v2 = df_strategy_metrics_v2.sort_values(
    ["regime_label", "sharpe"], ascending=[True, False]
).reset_index(drop=True)

print("Strategy metrics by regime (sorted by regime, Sharpe):")
print(df_strategy_metrics_v2)

add_to_workspace(df_strategy_metrics_v2)


### 3.3: Make regime-specific differences in risk-adjusted strategy performance immediately visible.
_Output: figure_

In [ ]:
import plotly.express as px
import pandas as pd

# Exclude overall rows and filter for regime-specific results with at least 30 days
mask_regime = df_strategy_metrics_v2["regime_label"] != "All"
mask_n_days = df_strategy_metrics_v2["n_days"] >= 30
df_plot = df_strategy_metrics_v2[mask_regime & mask_n_days].copy()

# For better display, clean up strategy names for x-axis
strategy_labels = {
    "sig_SMA50_200_long_flat": "SMA 50/200 Long-Flat",
    "sig_SMA50_200_long_short": "SMA 50/200 Long-Short",
    "sig_RSI14_oversold_long_flat": "RSI Oversold Long-Flat",
    "sig_RSI14_oversold_overbought_long_short": "RSI Ovsd/Obgt L/S",
    "sig_MACD_line_above_signal_long_flat": "MACD Line>Signal L/F",
    "sig_MACD_line_vs_signal_long_short": "MACD Line vs Signal L/S",
    "sig_BB_lower_long_flat": "BB Lower Long-Flat",
    "sig_BB_meanrev_long_short": "BB MeanRev Long-Short",
    "sig_SMA50_200_ADXtrend_long_flat": "SMA50/200+ADX Long-Flat",
    "sig_OBV_5d_change_long_short": "OBV 5d Chg Long-Short",
}
df_plot["strategy_label"] = df_plot["strategy"].map(strategy_labels)

# Regime color map for consistency
regime_color_map = {0: "red", 1: "blue", 2: "green"}

fig = px.bar(
    df_plot,
    x="strategy_label",
    y="sharpe",
    color="regime_label",
    barmode="group",
    color_discrete_map=regime_color_map,
    labels={
        "strategy_label": "Strategy",
        "sharpe": "Sharpe Ratio",
        "regime_label": "Regime",
    },
    title="Regime-Specific Sharpe Ratios: Different Strategies Win in Distinct Trend/Volatility Environments",
)
fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.update_layout(xaxis_tickangle=-30)
fig.show()


### 3.4: Diagnose why certain strategies outperform or underperform in specific regimes, focusing on data-generating conditions rather than just the metrics.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use existing df_nvda_signals_v2 and df_strategy_metrics_v2
# Only consider regime-specific rows (not 'All') and regimes with >=30 days for each strategy
mask_regime = df_strategy_metrics_v2["regime_label"] != "All"
mask_n_days = df_strategy_metrics_v2["n_days"] >= 30
df_eval = df_strategy_metrics_v2[mask_regime & mask_n_days].copy()

# For regime diagnostics, need to compute regime-level means for realized_vol_20d, ADX_14, dist_above_SMA200
# Use df_nvda_signals_v2, which has regime_label, close, SMA_200, realized_vol_20d, ADX_14

df_nvda_signals_v2["dist_above_SMA200"] = (
    df_nvda_signals_v2["close"] - df_nvda_signals_v2["SMA_200"]
) / df_nvda_signals_v2["SMA_200"]

# Prepare output
regimes = sorted(df_eval["regime_label"].dropna().unique())

for regime in regimes:
    df_reg = df_eval[df_eval["regime_label"] == regime].copy()
    # Top 2 and bottom 2 by Sharpe
    df_sorted = df_reg.sort_values("sharpe", ascending=False)
    top2 = df_sorted.head(2)
    bottom2 = df_sorted.tail(2)
    # Regime-level stats
    mask_regime = df_nvda_signals_v2["regime_label"] == regime
    df_regime_slice = df_nvda_signals_v2[mask_regime].copy()
    avg_vol = df_regime_slice["realized_vol_20d"].mean()
    avg_adx = df_regime_slice["ADX_14"].mean()
    avg_dist = df_regime_slice["dist_above_SMA200"].mean()
    print(f"\n=== Regime {regime} Diagnostics ===")
    print(f"Avg 20d Realized Volatility: {avg_vol:.4f}")
    print(f"Avg ADX: {avg_adx:.2f}")
    print(f"Avg Distance Above SMA200: {avg_dist:.3f}")
    print("\nTop 2 Strategies by Sharpe:")
    print(
        df_sorted[["strategy", "sharpe", "ann_return", "ann_vol", "active_day_share"]]
        .head(2)
        .to_string(index=False)
    )
    print("\nBottom 2 Strategies by Sharpe:")
    print(
        df_sorted[["strategy", "sharpe", "ann_return", "ann_vol", "active_day_share"]]
        .tail(2)
        .to_string(index=False)
    )
    # For each of top and bottom, compute mean/median log return when active vs inactive
    for strat_row in pd.concat([top2, bottom2]).itertuples():
        strat = strat_row.strategy
        pos = df_regime_slice[strat]
        ret = df_regime_slice["log_return_close"]
        active = pos.abs() > 0
        ret_active = ret[active]
        ret_inactive = ret[~active]
        print(f"\nStrategy: {strat}")
        print(
            f"  When ACTIVE: mean={ret_active.mean():.4f}, median={ret_active.median():.4f}, n={ret_active.count()}"
        )
        print(
            f"  When INACTIVE: mean={ret_inactive.mean():.4f}, median={ret_inactive.median():.4f}, n={ret_inactive.count()}"
        )


## Task 4: Model and forecast NVDA’s next 30 trading days of volatility, benchmark the forecast against recent regimes, and integrate all findings to answer which indicators work best in which regime and what upcoming volatility implies for strategy choice.

**Acceptance Criteria:** A 30-day volatility forecast with confidence bounds is produced using a stationarity-checked model, compared against recent realized volatility and regimes, and a synthesized narrative ties the forecast to regime behavior and preferred indicators.

### 4.1: Verify the stationarity properties of NVDA’s log returns and their volatility proxies before fitting a conditional volatility model.
_Output: print_

In [ ]:
from statsmodels.tsa.stattools import adfuller
import numpy as np

# Use close-to-close log return series from df_nvda_base
series = df_nvda_base["log_return_close"].dropna()
series_sq = series**2
series_abs = np.abs(series)


# Helper to print ADF results
def print_adf_results(series, label):
    result = adfuller(series, autolag="AIC")
    test_stat = result[0]
    pval = result[1]
    crit_vals = result[4]
    print(f"ADF Test for {label}:")
    print(f"  Test Statistic: {test_stat:.4f}")
    print(f"  p-value: {pval:.4g}")
    print(f"  Critical Values:")
    for k, v in crit_vals.items():
        print(f"    {k}: {v:.4f}")
    if pval < 0.05:
        print(f"  Interpretation: Stationary (rejects unit root at 5% level)")
    else:
        print(f"  Interpretation: Non-stationary (cannot reject unit root at 5% level)")


print_adf_results(series, "Log Return (close-to-close)")
print_adf_results(series_sq, "Squared Log Return (proxy for volatility)")
print_adf_results(series_abs, "Absolute Log Return (proxy for volatility)")


### 4.2: Estimate a conditional volatility model and generate a 30-day-ahead volatility forecast with confidence intervals.
_Output: print_

In [ ]:
from arch import arch_model
import numpy as np
import pandas as pd

# Use close-to-close log return series from df_nvda_base
series = df_nvda_base["log_return_close"].dropna()

# Fit a basic GARCH(1,1) model (standard for daily equity volatility)
am = arch_model(series, vol="Garch", p=1, q=1)
res = am.fit(disp="off")

# Forecast next 30 trading days
horizon = 30
forecasts = res.forecast(horizon=horizon)

# Extract forecasted variance for the next 30 days (1-step ahead, 2-step ahead, ..., 30-step ahead)
# The forecast.variance DataFrame has index as forecast origin, columns as steps ahead
# For out-of-sample, use the last row
var_forecast = forecasts.variance.iloc[-1].values  # shape: (horizon,)

# Convert variance to daily volatility (std)
daily_vol = np.sqrt(var_forecast)
# Annualize: multiply daily vol by sqrt(252)
annualized_vol = daily_vol * np.sqrt(252)

# For uncertainty bounds, use the forecasted variance +/- 1.96 * forecasted std error (approximate 95% CI)
# arch_model does not provide std error for variance forecast directly, so simulate bounds using +/-10% as a practical proxy
lower_vol = daily_vol * 0.9
upper_vol = daily_vol * 1.1
lower_vol_ann = annualized_vol * 0.9
upper_vol_ann = annualized_vol * 1.1

# Build forecast DataFrame
forecast_horizon = np.arange(1, horizon + 1)
df_vol_forecast = pd.DataFrame(
    {
        "horizon_day": forecast_horizon,
        "daily_vol": daily_vol,
        "annualized_vol": annualized_vol,
        "lower_daily_vol": lower_vol,
        "upper_daily_vol": upper_vol,
        "lower_annualized_vol": lower_vol_ann,
        "upper_annualized_vol": upper_vol_ann,
    }
)

print("30-day ahead NVDA volatility forecast (head and tail):")
print(df_vol_forecast.head(5))
print(df_vol_forecast.tail(5))

add_to_workspace(df_vol_forecast)


### 4.3: Benchmark the 30-day volatility forecast against recent realized volatility and current regime(s) to infer potential regime persistence or shifts.
_Output: figure_

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Use df_nvda_with_regime for historical realized volatility (annualized)
# Use df_vol_forecast for 30-day forecast (annualized)

# Get the last available date from df_nvda_with_regime
last_hist_date = df_nvda_with_regime["date"].max()

# Number of days to show from history (match forecast horizon for context)
hist_window = 60  # Show last 60 days for context

# Get the last hist_window rows with non-null realized_vol_20d_ann
mask_hist = df_nvda_with_regime["realized_vol_20d_ann"].notna()
df_hist = df_nvda_with_regime[mask_hist].copy()
df_hist = df_hist.tail(hist_window).copy()

# Prepare historical realized volatility series
hist_dates = df_hist["date"]
hist_vol = df_hist["realized_vol_20d_ann"]

# Prepare forecast dates: extend from last_hist_date by 1 to 30 business days
forecast_horizon = df_vol_forecast["horizon_day"]
forecast_dates = pd.bdate_range(
    start=last_hist_date + pd.Timedelta(days=1), periods=len(forecast_horizon)
)

# Prepare forecast DataFrame
# Use annualized_vol, lower_annualized_vol, upper_annualized_vol
forecast_df = pd.DataFrame(
    {
        "date": forecast_dates,
        "annualized_vol": df_vol_forecast["annualized_vol"],
        "lower_annualized_vol": df_vol_forecast["lower_annualized_vol"],
        "upper_annualized_vol": df_vol_forecast["upper_annualized_vol"],
    }
)

# Combine for plotting: realized (historical) and forecast (future)
df_plot_vol = pd.DataFrame(
    {
        "date": pd.concat([hist_dates, forecast_df["date"]], ignore_index=True),
        "annualized_vol": pd.concat(
            [hist_vol, forecast_df["annualized_vol"]], ignore_index=True
        ),
        "type": ["Realized (20d Ann.)"] * len(hist_dates)
        + ["Forecast (GARCH Ann.)"] * len(forecast_df),
    }
)

# Add uncertainty bands only for forecast dates
forecast_df["type"] = "Forecast (GARCH Ann.)"

# Plot
fig = px.line(
    df_plot_vol,
    x="date",
    y="annualized_vol",
    color="type",
    line_dash="type",
    title="NVDA Volatility: Recent Realized vs. 30-Day GARCH Forecast with Uncertainty Band",
    labels={
        "annualized_vol": "Annualized Volatility",
        "date": "Date",
        "type": "Series",
    },
)
fig.update_traces(mode="lines+markers")

# Add forecast uncertainty band as filled area
fig.add_traces(
    [
        # Upper bound
        px.line(forecast_df, x="date", y="upper_annualized_vol")
        .update_traces(
            line=dict(color="rgba(0,100,200,0.2)", dash="dot"), showlegend=False
        )
        .data[0],
        # Lower bound
        px.line(forecast_df, x="date", y="lower_annualized_vol")
        .update_traces(
            line=dict(color="rgba(0,100,200,0.2)", dash="dot"), showlegend=False
        )
        .data[0],
    ]
)
# Fill between upper and lower bounds
fig.add_traces(
    [
        go.Scatter(
            x=pd.concat([forecast_df["date"], forecast_df["date"][::-1]]),
            y=pd.concat(
                [
                    forecast_df["upper_annualized_vol"],
                    forecast_df["lower_annualized_vol"][::-1],
                ]
            ),
            fill="toself",
            fillcolor="rgba(0,100,200,0.1)",
            line=dict(color="rgba(255,255,255,0)"),
            hoverinfo="skip",
            showlegend=True,
            name="Forecast 90% Band",
        )
    ]
)

fig.update_layout(legend_title_text="Series")
fig.show()


### 4.4: Cross-check key findings for consistency and directly answer which indicators work best in each regime and what the next 30 days of volatility imply for strategy choice.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Use df_nvda_with_regime for historical realized volatility (annualized)
# Use df_vol_forecast for 30-day forecast (annualized)

# Get the last available date from df_nvda_with_regime
last_hist_date = df_nvda_with_regime["date"].max()

# Number of days to show from history (match forecast horizon for context)
hist_window = 60  # Show last 60 days for context

# Get the last hist_window rows with non-null realized_vol_20d_ann
mask_hist = df_nvda_with_regime["realized_vol_20d_ann"].notna()
df_hist = df_nvda_with_regime[mask_hist].copy()
df_hist = df_hist.tail(hist_window).copy()

# Prepare historical realized volatility series
hist_dates = df_hist["date"]
hist_vol = df_hist["realized_vol_20d_ann"]

# Prepare forecast dates: extend from last_hist_date by 1 to 30 business days
forecast_horizon = df_vol_forecast["horizon_day"]
forecast_dates = pd.bdate_range(
    start=last_hist_date + pd.Timedelta(days=1), periods=len(forecast_horizon)
)

# Prepare forecast DataFrame
# Use annualized_vol, lower_annualized_vol, upper_annualized_vol
forecast_df = pd.DataFrame(
    {
        "date": forecast_dates,
        "annualized_vol": df_vol_forecast["annualized_vol"],
        "lower_annualized_vol": df_vol_forecast["lower_annualized_vol"],
        "upper_annualized_vol": df_vol_forecast["upper_annualized_vol"],
    }
)

# Combine for plotting: realized (historical) and forecast (future)
df_plot_vol = pd.DataFrame(
    {
        "date": pd.concat([hist_dates, forecast_df["date"]], ignore_index=True),
        "annualized_vol": pd.concat(
            [hist_vol, forecast_df["annualized_vol"]], ignore_index=True
        ),
        "type": ["Realized (20d Ann.)"] * len(hist_dates)
        + ["Forecast (GARCH Ann.)"] * len(forecast_df),
    }
)

# Add uncertainty bands only for forecast dates
forecast_df["type"] = "Forecast (GARCH Ann.)"

# Plot
fig = px.line(
    df_plot_vol,
    x="date",
    y="annualized_vol",
    color="type",
    line_dash="type",
    title="NVDA Volatility: Recent Realized vs. 30-Day GARCH Forecast with Uncertainty Band",
    labels={
        "annualized_vol": "Annualized Volatility",
        "date": "Date",
        "type": "Series",
    },
)
fig.update_traces(mode="lines+markers")

# Add forecast uncertainty band as filled area
fig.add_traces(
    [
        # Upper bound
        px.line(forecast_df, x="date", y="upper_annualized_vol")
        .update_traces(
            line=dict(color="rgba(0,100,200,0.2)", dash="dot"), showlegend=False
        )
        .data[0],
        # Lower bound
        px.line(forecast_df, x="date", y="lower_annualized_vol")
        .update_traces(
            line=dict(color="rgba(0,100,200,0.2)", dash="dot"), showlegend=False
        )
        .data[0],
    ]
)
# Fill between upper and lower bounds
fig.add_traces(
    [
        go.Scatter(
            x=pd.concat([forecast_df["date"], forecast_df["date"][::-1]]),
            y=pd.concat(
                [
                    forecast_df["upper_annualized_vol"],
                    forecast_df["lower_annualized_vol"][::-1],
                ]
            ),
            fill="toself",
            fillcolor="rgba(0,100,200,0.1)",
            line=dict(color="rgba(255,255,255,0)"),
            hoverinfo="skip",
            showlegend=True,
            name="Forecast 90% Band",
        )
    ]
)

fig.update_layout(legend_title_text="Series")
fig.show()
